# Kombinierte Labels: Erweiterte RBF-SVM-Suche

**Hypothese:** Das Optimum der bisherigen RBF-SVM-Suche lag bei `C=10` und `gamma=0.01` am Rand des Suchraums. Eine gezielte Erweiterung könnte eine höhere kombinierte Balanced Accuracy erreichen.

**Ablauf:** Die Auswahl erfolgt ausschließlich mit fünf festen Cross-Validation-Folds auf dem Trainingsanteil. Der gemeinsame Validierungssplit bleibt bis zur einmaligen Bewertung des besten CV-Kandidaten unberührt.

In [ ]:
%load_ext autoreload
%autoreload 2

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

svm_preprocessing = PreprocessingConfig(scale=True, use_meta=True)

## 1. Hyperparameter suchen

Die bisherige Suche umfasste `C` bis 10 und `gamma` bis 0.01; der Gewinner lag an beiden Rändern. Diese Suche erweitert beide Dimensionen kontrolliert auf zwölf Kandidaten. Die Rangfolge bestimmt ausschließlich `bacc_combined` in der Cross-Validation.

In [ ]:
tuned = tune(
    make_combined_svm(),
    TuneConfig(
        name="combined_svm_expanded_search",
        approach="combined",
        description=(
            "Erweiterte RBF-SVM-Suche auf kombinierten Crop/Stage-Labels, "
            "weil der bisherige CV-Gewinner am Rand von C und gamma lag."
        ),
        preprocessing=svm_preprocessing,
        param_grid={
            "estimator__C": [3.0, 10.0, 30.0, 100.0],
            "estimator__gamma": [0.003, 0.01, 0.03],
        },
    ),
)

tuned.candidates.sort_values("rank")

## 2. Einmal validieren

Der CV-Gewinner wird einmal auf dem gemeinsamen Validierungssplit bewertet. Dieser Score wird nicht für weitere Änderungen am Suchraum verwendet.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="combined_svm_expanded",
        approach="combined",
        description=(
            "Beste RBF-SVM aus combined_svm_expanded_search, einmal auf dem "
            "gemeinsamen Validierungssplit bewertet."
        ),
        preprocessing=svm_preprocessing,
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## 3. Analyse

Die erweiterte Suche wird gegen die bisherige SVM-Referenz eingeordnet. Neben der kombinierten Balanced Accuracy zeigen die Abbildungen, ob bestimmte Kulturen oder Stadien weiterhin besonders häufig verwechselt werden.

In [ ]:
from IPython.display import display

from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

REFERENCE_SVM_CV_BACC_COMBINED = 0.83382
REFERENCE_SVM_VALIDATION_BACC_COMBINED = 0.845

comparison = {
    "CV BAcc kombiniert": {
        "RBF-SVM Referenz": REFERENCE_SVM_CV_BACC_COMBINED,
        "Erweiterte RBF-SVM": tuned.best_score,
    },
    "Validierung BAcc kombiniert": {
        "RBF-SVM Referenz": REFERENCE_SVM_VALIDATION_BACC_COMBINED,
        "Erweiterte RBF-SVM": result.metrics.bacc_combined,
    },
}
display(comparison)

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Kombinierte Labels: Erweiterte CV-Suche der RBF-SVM")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

In [ ]:
{
    "tuning_run": tuned.run_id,
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "evaluation_run": result.run_id,
}

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Kombinierte Labels: Erweiterte CV-Suche der RBF-SVM")
save_doc_figure(tuning_ax.figure, "combined_svm_expanded_tuning", MODEL_DOCS_FIGURES_DIR)

confusion_figure = plot_confusion_matrices(result.y_val, result.y_pred)
save_doc_figure(confusion_figure, "combined_svm_expanded_confusion", MODEL_DOCS_FIGURES_DIR)

recall_figure = plot_class_recall(result.y_val, result.y_pred)
save_doc_figure(recall_figure, "combined_svm_expanded_recall", MODEL_DOCS_FIGURES_DIR)

confusions_figure = plot_top_confusions(result.y_val, result.y_pred)
save_doc_figure(confusions_figure, "combined_svm_expanded_confusions", MODEL_DOCS_FIGURES_DIR)

## Fazit

Die erweiterte Suche verbessert die CV-BAcc kombiniert gegenüber der bisherigen RBF-SVM-Referenz. Der einmalige Validierungslauf entscheidet, ob diese Konfiguration als neue Referenz dokumentiert wird; weitere SVM-Parameter werden nicht anhand dieses Holdout-Ergebnisses gesucht.

## 4. Breitere CV-Sensitivitätsanalyse

Die Validierung wurde für die erweiterte Suche bereits einmal verwendet. Diese zweite Suche bewertet deshalb ausschließlich die fünf Trainingsfolds und verändert weder die Referenz noch die dokumentierten Holdout-Metriken.

Der bisherige CV-Gewinner lag bei `C=100.0` und `gamma=0.003`. Die Suche erweitert `C` nach oben und `gamma` nach unten auf einer logarithmischen Skala. `class_weight="balanced"`, der RBF-Kernel und die Vorverarbeitung bleiben unverändert, weil sie Teil der geprüften Referenzkonfiguration sind. `tol`, `shrinking`, `cache_size` und `max_iter` beeinflussen nur Konvergenz oder Laufzeit und werden nicht nach dem Score optimiert.

In [ ]:
wide_tuned = tune(
    make_combined_svm(),
    TuneConfig(
        name="combined_svm_wide_cv_search",
        approach="combined",
        description=(
            "CV-only Sensitivitätsanalyse der RBF-SVM nach der einmaligen Holdout-Bewertung "
            "von combined_svm_expanded; erweitert C nach oben und gamma nach unten."
        ),
        preprocessing=svm_preprocessing,
        param_grid={
            "estimator__C": [10.0, 30.0, 100.0, 300.0, 1000.0],
            "estimator__gamma": [0.0001, 0.0003, 0.001, 0.003, 0.01],
        },
    ),
)

wide_tuned.candidates.sort_values("rank")

In [ ]:
wide_tuning_ax = plot_tuning_candidates(
    wide_tuned.candidates,
    color_by="estimator__C",
)
wide_tuning_ax.set_title("Kombinierte Labels: Breite CV-Sensitivitätsanalyse der RBF-SVM")
display(wide_tuning_ax.figure)

Die breite Suche erreicht ihr CV-Maximum bei `C=300.0` und `gamma=0.001` mit einer kombinierten BAcc von 0,852. Der Vorsprung von 0,004 gegenüber `C=100.0`, `gamma=0.003` ist kleiner als die Streuung zwischen den Folds. Weil der gemeinsame Holdout bereits für die bestehende Referenz verwendet wurde, wird keine zweite Validierung ausgeführt und die Referenz nicht geändert.